In [8]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch 
import time 
import evaluate  ## for calculating rouge score
import pandas as pd
import numpy as np

# Loading dataset

In [9]:
## dataset from https://huggingface.co/datasets/knkarthick/dialogsum
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset

Using the latest cached version of the dataset since knkarthick/dialogsum couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\utsav\.cache\huggingface\datasets\knkarthick___dialogsum\default\0.0.0\a968e7aee0602e257935f1321a02e4287f7d5848 (last modified on Tue Jan 21 03:29:40 2025).


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [10]:
dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Subset of Main Data

In [11]:
from datasets import load_dataset, DatasetDict

# Load your original dataset
#dataset = load_dataset('your_dataset_name')  # replace 'your_dataset_name' with the actual dataset name

# Select the specified number of examples from each split
train_subset = dataset['train'].select(range(30))
validation_subset = dataset['validation'].select(range(5))
test_subset = dataset['test'].select(range(5))

# Create a new DatasetDict with the smaller subsets
small_dataset = DatasetDict({
    'train': train_subset,
    'validation': validation_subset,
    'test': test_subset
})

small_dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Load LLM

In [12]:
## for more information of model https://huggingface.co/google/flan-t5-base
model_name = 'google/flan-t5-base'
#model_name = 't5-small'

# bfloat16 mean we are using the small version of flan-t5
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Trainable Parameter

In [13]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f'trainable model parameters: {trainable_model_params}\n \
            all model parameters: {all_model_params} \n \
            percentage of trainable model parameters: {(trainable_model_params / all_model_params) * 100} %'


print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
             all model parameters: 247577856 
             percentage of trainable model parameters: 100.0 %


# data preprocessing

In [14]:
def tokeninze_function(example):
    start_prompt = 'Summarize the following conversation. \n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]
    #print(prompt)
    example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, 
                                     return_tensors='pt').input_ids
    example['labels'] = tokenizer(example['summary'], padding='max_length', truncation=True, 
                                 return_tensors='pt').input_ids
    
    return example

# The Dataseta ctually contains 3 diff splits: train, validation, and test.
# The tokenize_function code is handling all data across all splits in batches
tokenize_datasets = small_dataset.map(tokeninze_function, batched=True)
tokenize_datasets = tokenize_datasets.remove_columns(['id', 'topic', 'dialogue',
                                                     'summary'])

Map: 100%|████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 98.88 examples/s]


In [16]:
tokenize_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 30
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
})

# USE PEFT Configuration

In [17]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(r=32, #rank 32,
                         lora_alpha=32, ## LoRA Scaling factor 
                         target_modules=['q', 'v'], ## The modules(for example, attention blocks) to apply the LoRA update matrices.
                         lora_dropout = 0.05,
                         bias='none',
                         task_type=TaskType.SEQ_2_SEQ_LM ## flan-t5
)

## target_modules='q', This represents the value projection layer in the transformer model. The value projection layer transforms input tokens into value vectors,
# which are the actual values that are attended to based on the attention scores computed from query and key vectors.

## target_modules='v',This typically refers to the query projection layer in a transformer-based model. The query projection layer is responsible for transforming 
# input tokens into query vectors, which are used to attend to other tokens in the sequence during self-attention mechanism.

In [18]:
peft_model = get_peft_model(original_model, lora_config)

print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 3538944
             all model parameters: 251116800 
             percentage of trainable model parameters: 1.4092820552029972 %


# Train-peft-model

In [19]:
#output_dir = f'./dialogue-summary-training-{str(int(time.time()))}'
## this is we are again back to the hugging face trainer module

peft_training_args = TrainingArguments(output_dir="trainer-api-logs-2025",
                                       auto_find_batch_size=True,
                                       evaluation_strategy="epoch",
                                       learning_rate=1e-3,
                                       num_train_epochs=1,
                                       logging_steps=1,
                                       max_steps=1,
                                       report_to='none'
                                       #load_best_model_at_end=True
                                                           
                )

## this is same except we are using PEFT model instead of regular
peft_trainer = Trainer(model=peft_model, 
                      args=peft_training_args,
                      train_dataset=tokenize_datasets['train'],
                      eval_dataset=tokenize_datasets['validation'] 
                 )


peft_trainer.train()

peft_model_path = 'peft-model-path-2025'

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)


C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


Epoch,Training Loss,Validation Loss
0,49.750000,51.250000


C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('peft-model-path-2025\\tokenizer_config.json',
 'peft-model-path-2025\\special_tokens_map.json',
 'peft-model-path-2025\\spiece.model',
 'peft-model-path-2025\\added_tokens.json',
 'peft-model-path-2025\\tokenizer.json')

In [21]:
from peft import PeftModel
peft_model_base = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base', torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

peft_model = PeftModel.from_pretrained(peft_model_base, 
                                      'peft-model-path-2025',
                                      torch_dtype=torch.bfloat16,
                                      is_trainable=False) ## is_trainable mean just a forward pass jsut to get a sumamry

index = 6 ## randomly pick index
dialogue = dataset['test'][index]['dialogue']
human_baseline_summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

input_ids = tokenizer(prompt, return_tensors='pt').input_ids

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)


peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

print(f'Human Baseline summary: \n{human_baseline_summary}\n')
print(f'Original Model Output \n{original_model_text_output}\n')
print(f'Peft Model Output \n{peft_model_text_output}\n')

Human Baseline summary: 
#Person1# tells Kate that Masha and Hero get divorced. Kate is surprised because she thought they are perfect couple.

Original Model Output 
Masha and Hero are getting divorced................................................................................................

Peft Model Output 
Masha and Hero are getting divorced................................................................................................



In [ ]:
 dataset['test'][6]